# AgriSense - Outlier Detection for Yield &amp; Price Data

Welcome to Section 4.43! In this notebook, we will learn how to detect and handle **Outliers** in our agricultural data before we train our Machine Learning models.

### What is an Outlier?
An outlier is a data point that differs significantly from other observations. In agriculture, outliers can occur due to:
*   **Data Entry Errors:** Someone accidentally typed a yield of 5000 instead of 50.
*   **Extreme Weather Events:** A massive flood causing 500mm of unexpected rainfall.
*   **Market Shocks:** A sudden shortage causing onion prices to skyrocket by 400% in a week.

### Why do we care?
Machine Learning models (like Linear Regression for Yield Prediction) are highly sensitive to outliers. If we train our model on extreme, unrealistic data, it will make poor predictions for normal, day-to-step scenarios.

### Detection Methods we will cover:
1.  **Visual Method:** Identifying outliers using Boxplots and Scatter plots.
2.  **Statistical Method (IQR):** The Interquartile Range rule (1.5 * IQR).
3.  **Statistical Method (Z-Score):** Measuring how many standard deviations away a point is.

In [ ]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Suppress minor warnings for cleaner output
warnings.filterwarnings('ignore')

# Create output directory for saving outlier figures
Path("../outputs/figures/outliers").mkdir(parents=True, exist_ok=True)

# Set a professional visual style
sns.set_theme(style="whitegrid")
print("Libraries imported successfully! Ready to find outliers.")

## 1. Load Data &amp; Inject Realistic Agricultural Outliers
Our `mandi_prices.csv` is quite clean. To practice outlier detection, we will load it and simulate realistic agricultural features (`yield`, `rainfall`, `temperature`) and intentionally introduce some absolute **data disasters** (outliers).

In [ ]:
# Load the dataset
data_path = Path("../data/raw/mandi_prices.csv")
df = pd.read_csv(data_path)
df.columns = df.columns.str.lower().str.strip()

# Generate realistic base data for environmental features
np.random.seed(42)
df['yield'] = np.random.normal(loc=30, scale=5, size=len(df))        # Normal yield around 30 q/ha
df['rainfall'] = np.random.normal(loc=120, scale=20, size=len(df))   # Normal rainfall around 120 mm
df['temperature'] = np.random.normal(loc=28, scale=4, size=len(df))  # Normal temp around 28°C

# INJECT REALISTIC OUTLIERS (The "Data Disasters")
# 1. Unrealistic high price (e.g., typing error: an extra zero)
df.loc[5, 'modal_price'] = df['modal_price'].mean() * 5  

# 2. Impossible 0 yield (crop failure or data missing)
df.loc[12, 'yield'] = 0.0  

# 3. Extreme Rainfall (Flood event)
df.loc[25, 'rainfall'] = 450.0  

# 4. Extreme Heatwave
df.loc[30, 'temperature'] = 48.0  

print("Data loaded and outliers injected!")
print("Here is a quick statistical summary showing the skewed maximums:")
display(df[['modal_price', 'yield', 'rainfall', 'temperature']].describe().round(1))

## 2. Visual Detection Method: Boxplots
The quickest way to spot outliers is visualization. Boxplots automatically represent outliers as individual dots sitting outside the "whiskers" of the box.

In [ ]:
def plot_boxplots(data, title, filename):
    """Reusable function to plot boxplots for numerical columns."""
    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    
    sns.boxplot(y=data['modal_price'], ax=axes[0], color='skyblue')
    axes[0].set_title('Modal Price (₹)')
    
    sns.boxplot(y=data['yield'], ax=axes[1], color='lightgreen')
    axes[1].set_title('Yield (q/ha)')
    
    sns.boxplot(y=data['rainfall'], ax=axes[2], color='lightcyan')
    axes[2].set_title('Rainfall (mm)')
    
    sns.boxplot(y=data['temperature'], ax=axes[3], color='lightcoral')
    axes[3].set_title('Temperature (°C)')
    
    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.savefig(f'../outputs/figures/outliers/{filename}', dpi=300)
    plt.show()

# Visualize the data WITH outliers
plot_boxplots(df, "Visualizing Outliers in Raw Data", "before_outliers.png")

**💡 Insight:** 
Look at those single dots sitting way above (or below) the colored boxes! 
* That ₹9,000+ price spike is clearly an error.
* That 450mm rainfall is a massive flood. 

## 3. Statistical Detection Method: The IQR Rule
The Interquartile Range (IQR) represents the middle 50% of the data (between the 25th and 75th percentiles).
Any value outside the boundary of `1.5 * IQR` is mathematically considered an outlier.

Let's build a reusable function to detect these.

In [ ]:
def detect_outliers_iqr(df, column):
    """
    Detect outliers using the IQR method.
    Returns: The lower bound, upper bound, and a dataframe of just the outliers.
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Filter the dataframe to find rows outside the bounds
    outliers = df[(df[column] &lt; lower_bound) | (df[column] &gt; upper_bound)]
    
    print(f"--- Outlier Detection for '{column}' ---")
    print(f"Lower Boundary: {lower_bound:.2f}")
    print(f"Upper Boundary: {upper_bound:.2f}")
    print(f"Number of Outliers Found: {len(outliers)}")
    print(f"Percentage of Data: {(len(outliers)/len(df))*100:.1f}%\n")
    
    return lower_bound, upper_bound, outliers

# Test the function on the 'modal_price' column
lower_p, upper_p, outliers_price = detect_outliers_iqr(df, 'modal_price')
display(outliers_price[['commodity', 'state', 'modal_price']])

## 4. Handling Strategies
Now that we found them, what do we do? We have 3 main strategies:

### Strategy A: Removal (Dropping)
Best used when you know the data is physically impossible (like a typing error or an empty record).
Let's remove the unrealistic price outlier.

In [ ]:
# Strategy A: Removal
# We will keep only the rows where modal_price is strictly BETWEEN the bounds
df_cleaned = df[(df['modal_price'] &gt;= lower_p) &amp; (df['modal_price'] &lt;= upper_p)].copy()

print(f"Original Data Rows: {len(df)}")
print(f"Cleaned Data Rows: {len(df_cleaned)}")
print(f"Max Price before: ₹{df['modal_price'].max():.2f}")
print(f"Max Price after:  ₹{df_cleaned['modal_price'].max():.2f}")

### Strategy B: Capping / Winsorization
Instead of deleting valuable rows of data, we "cap" the extreme values at the upper or lower boundary. 
Best used for continuous environmental data like Rainfall. A flood happened, but we cap it at our upper boundary so it doesn't break our Machine Learning model.

In [ ]:
# Strategy B: Capping
lower_r, upper_r, _ = detect_outliers_iqr(df_cleaned, 'rainfall')

# Replace any values above the upper bound WITH the upper bound
df_cleaned['rainfall'] = np.where(df_cleaned['rainfall'] &gt; upper_r, upper_r, df_cleaned['rainfall'])

# Replace any values below the lower bound WITH the lower bound
df_cleaned['rainfall'] = np.where(df_cleaned['rainfall'] &lt; lower_r, lower_r, df_cleaned['rainfall'])

print(f"Max Rainfall after Capping: {df_cleaned['rainfall'].max():.2f} mm (Capped at Boundary)")

### Strategy C: Flagging (Keeping them, but marking them)
Sometimes extreme values are highly important to predict (like crop failures). We keep the value but add an `is_outlier` column so our ML model knows there is a special circumstance.

In [ ]:
def flag_outliers(df, column):
    """Adds a boolean (True/False) flag column for outliers."""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Create the flag column
    flag_col_name = f'is_{column}_outlier'
    df[flag_col_name] = ((df[column] &lt; lower_bound) | (df[column] &gt; upper_bound)).astype(int)
    
    print(f"Flagged {df[flag_col_name].sum()} rows as {column} outliers.")
    return df

# Flag extreme low/high yields (like crop failures) instead of removing them
df_cleaned = flag_outliers(df_cleaned, 'yield')

# Show the row where yield was 0
display(df_cleaned[df_cleaned['yield'] == 0.0][['commodity', 'yield', 'is_yield_outlier']])

## 5. Result: Before vs. After
Let's see how our data looks after applying these strategies.

In [ ]:
# Visualize the CLEANED data side-by-side
plot_boxplots(df_cleaned, "Outliers Handled (Cleaned Data)", "after_outliers.png")

In [ ]:
print("--- SUMMARY STATISTICS COMPARISON (Mean values) ---")
print(f"Average Price Before: ₹{df['modal_price'].mean():.2f}   | After: ₹{df_cleaned['modal_price'].mean():.2f}")
print(f"Average Rain  Before: {df['rainfall'].mean():.2f} mm | After: {df_cleaned['rainfall'].mean():.2f} mm")

## Conclusion &amp; Next Steps

### How does this improve our Yield Prediction Model?
1. **Removes Model Distraction:** By capping rainfall and removing typo-based price spikes, our future ML model (like Linear Regression) will not be "distracted" or skewed by one wildly inaccurate data point.
2. **Maintains Value:** By using flagging (`is_yield_outlier`), we taught the dataset to warn the model about legitimate extreme cases (like crop failures) without entirely deleting the record.

**Up Next:** 
Now that our data is clean, filtered, standardized, and free of extreme outliers, we are finally ready for **Feature Engineering** (creating new mathematical features from existing ones) to supercharge our Machine Learning algorithms!